# Unidirectional Smart EV Charging

GitHub-ready version of the unidirectional smart-charging model.

## 1. Imports

In [ ]:
import os
from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Optional interactive widgets (used in Colab/Jupyter, not required by GitHub)
try:
    import ipywidgets as widgets
    from IPython.display import display
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

# Enable Colab widgets when running in Google Colab
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass


## 2. Data and PV model

In [ ]:
def read_excel_data(filename, sheet_name):
    data = pd.read_excel(filename, sheet_name=sheet_name, header=None)
    return data.values


def find_input_file():
    """Find the Excel input file in Colab, local Jupyter, or the GitHub repo structure."""
    candidates = [
        Path("TP-V2G-Input_data.xlsx"),
        Path("/content/TP-V2G-Input_data.xlsx"),
        Path("../data/TP-V2G-Input_data.xlsx"),
        Path("data/TP-V2G-Input_data.xlsx"),
    ]

    for path in candidates:
        if path.exists():
            return path

    raise FileNotFoundError(
        "TP-V2G-Input_data.xlsx was not found. "
        "Upload it to Colab or place it in the repository data/ folder."
    )


def Mod_PVprod(Ta, E, Ppv_nom):
    """Algebraic photovoltaic production model."""
    NOCT = 48.0
    E_ref = 1000.0
    Tpv_ref = 25.0
    alpha = -4.8e-3

    Tpv = Ta + E * (NOCT - 20.0) / 800.0
    Ppv = (E / E_ref) * (Ppv_nom + alpha * (Tpv - Tpv_ref))

    return np.maximum(Ppv, 0.0)


## 3. Parameters

In [ ]:
# Problem parameters
deg2K = 273.15
kW2W = 1000.0
kWh2Wh = 1000.0
h2min = 60

N_ev = 50
capa_bat = 52          # Battery capacity [kWh]
d_autonomie = 320      # Vehicle range [km]
P1_ch = 22             # Normal charging power [kW]
vmoy = 40 / h2min      # Average speed [km/min]

Ppv_nom = 500e3        # Nominal PV capacity [W]

# Simulation period
j = 174
Nj = 7
ind_0 = j * 24 + 1
ind_f = (j + Nj) * 24 + 1


## 4. Load input data

In [ ]:
filename = find_input_file()

sheet_name = "Demands"
demands = read_excel_data(filename, sheet_name)

Hd = np.array(demands[1 + ind_0:1 + ind_f, 1].tolist()) * kW2W
Pd = np.array(demands[1 + ind_0:1 + ind_f, 2].tolist()) * kW2W
v_tps_h = np.array(demands[1 + ind_0:1 + ind_f, 0].tolist()) * h2min

sheet_name = "Meteo"
meteo = read_excel_data(filename, sheet_name)

E = np.array(meteo[1 + ind_0:1 + ind_f, 1].tolist())
Ta = np.array(meteo[1 + ind_0:1 + ind_f, 2].tolist())

# Interpolate to one-minute time steps
v_tps_mn = np.arange(v_tps_h[0], v_tps_h[-1], 1)

Pd_mn = np.interp(v_tps_mn, v_tps_h, Pd)
E_mn = np.interp(v_tps_mn, v_tps_h, E)
Ta_mn = np.interp(v_tps_mn, v_tps_h, Ta)

# PV production
Ppv = Mod_PVprod(Ta_mn, E_mn, Ppv_nom)

print(f"Loaded {len(v_tps_mn):,} one-minute time steps.")
print(f"Simulation horizon: {Nj} days.")


## 5. Unidirectional charging model

In [ ]:
def simulate_unidirectional(cap_factor_weekday=0.5, k_need_weekday=0.5, seed=174):
    """
    Simulate rule-based unidirectional EV charging.

    EV power is positive during charging.
    Charging is prioritized when PV generation is available.
    """
    P_ev = np.zeros(len(Pd_mn))

    cons_kWh_km = capa_bat / d_autonomie
    Pmax_W = P1_ch * kW2W
    dt_h = 1.0 / 60.0
    soc_min = 0.10
    soc_max = 1.00

    rng = np.random.default_rng(seed)

    # All EVs start at 100% SOC.
    E_bat = np.ones(N_ev) * (soc_max * capa_bat)

    # The series starts around 01:00, so recover the first midnight.
    t0_midnight = v_tps_mn[0] - 60

    def is_weekend(day_from_start):
        return day_from_start >= 5

    for d in range(Nj):
        day_midnight = t0_midnight + d * 1440
        i_day0 = np.searchsorted(v_tps_mn, day_midnight, side="left")
        i_day1 = np.searchsorted(v_tps_mn, day_midnight + 1440, side="left")

        if i_day1 <= i_day0:
            continue

        weekend = is_weekend(d)
        cap_factor = 1.0 if weekend else cap_factor_weekday

        # 1) Daily mobility
        if weekend:
            E_day = np.zeros(N_ev)
            arr_idx = np.full(N_ev, i_day0)
            dep_idx = np.full(N_ev, i_day1)
        else:
            hdep_home = rng.triangular(7.5, 8.5, 9.5, size=N_ev)
            d_traj = rng.triangular(10.0, 30.0, 50.0, size=N_ev)

            t_traj_h = d_traj / (vmoy * 60.0)
            harr_work = hdep_home + t_traj_h

            hdep_work = rng.triangular(17.0, 18.0, 19.0, size=N_ev)
            hdep_work = np.maximum(hdep_work, harr_work + 0.25)

            E_day = 2.0 * d_traj * cons_kWh_km

            arr_min = day_midnight + np.round(harr_work * 60.0).astype(int)
            dep_min = day_midnight + np.round(hdep_work * 60.0).astype(int)

            arr_idx = np.searchsorted(v_tps_mn, arr_min, side="left")
            dep_idx = np.searchsorted(v_tps_mn, dep_min, side="left")

        # Mobility energy consumption
        E_bat -= E_day
        E_bat = np.clip(E_bat, soc_min * capa_bat, soc_max * capa_bat)

        # 2) Charging target
        if weekend:
            E_need = (soc_max * capa_bat) - E_bat
        else:
            E_need = k_need_weekday * E_day

        E_need = np.maximum(E_need, 0.0)
        E_rem = E_need.copy()

        # 3) Minute-by-minute PV-following charging
        for t in range(i_day0, i_day1):
            pv_W = max(Ppv[t], 0.0)

            if pv_W <= 0:
                continue

            P_total_allowed = cap_factor * pv_W

            if P_total_allowed <= 0:
                continue

            plugged = (
                (t >= arr_idx)
                & (t < dep_idx)
                & (E_rem > 1e-9)
                & (E_bat < soc_max * capa_bat - 1e-9)
            )

            active = np.where(plugged)[0]

            if active.size == 0:
                continue

            # Maximum power allowed for each active vehicle
            P_need_W = (E_rem[active] / dt_h) * 1000.0
            P_cap_i = np.minimum(Pmax_W, P_need_W)

            # Water-filling allocation
            P_alloc = np.zeros(active.size)
            remaining = P_total_allowed
            alive = np.ones(active.size, dtype=bool)

            while remaining > 1e-6 and alive.any():
                idx_local = np.where(alive)[0]
                share = remaining / idx_local.size

                give = np.minimum(
                    share,
                    P_cap_i[idx_local] - P_alloc[idx_local]
                )

                P_alloc[idx_local] += give
                remaining -= give.sum()

                alive[idx_local] = (
                    P_alloc[idx_local]
                    < P_cap_i[idx_local] - 1e-9
                )

            P_ev[t] += P_alloc.sum()

            dE = (P_alloc / 1000.0) * dt_h

            E_rem[active] = np.maximum(
                E_rem[active] - dE,
                0.0
            )

            E_bat[active] = np.clip(
                E_bat[active] + dE,
                soc_min * capa_bat,
                soc_max * capa_bat
            )

    return P_ev


## 6. Evaluation metrics

In [ ]:
def smoothing_index(P_grid):
    """Load-smoothing indicator J = mean((ΔP_grid)^2)."""
    dP = np.diff(P_grid)
    return float(np.mean(dP ** 2))


def evaluate_unidirectional(cap_factor=0.30, k_need=0.80):
    P_ev = simulate_unidirectional(
        cap_factor_weekday=cap_factor,
        k_need_weekday=k_need
    )

    P_grid = Pd_mn + P_ev - Ppv

    metrics = {
        "smoothing_index": smoothing_index(P_grid),
        "peak_grid_kW": float(np.max(P_grid) / 1000.0),
        "charging_energy_kWh": float(
            np.sum(np.maximum(P_ev, 0.0)) / 1000.0 / 60.0
        ),
    }

    return P_ev, P_grid, metrics


## 7. Interactive exploration (Colab/Jupyter)

In [ ]:
def plot_scenario(cap_factor=0.50, k_need=0.50):
    P_ev, P_grid, metrics = evaluate_unidirectional(
        cap_factor=cap_factor,
        k_need=k_need
    )

    time_h = np.arange(len(Pd_mn)) / 60.0

    plt.figure(figsize=(12, 5))
    plt.plot(time_h, Pd_mn / 1000.0, label="Base demand")
    plt.plot(time_h, Ppv / 1000.0, label="PV generation")
    plt.plot(time_h, P_ev / 1000.0, label="EV charging")
    plt.plot(time_h, P_grid / 1000.0, label="Grid power")

    plt.title(
        f"Unidirectional Smart Charging — "
        f"cap_factor={cap_factor:.2f}, k_need={k_need:.2f}"
    )
    plt.xlabel("Time from start [h]")
    plt.ylabel("Power [kW]")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    print("---- Metrics ----")
    for key, value in metrics.items():
        print(f"{key}: {value}")


if WIDGETS_AVAILABLE:
    cap_slider = widgets.FloatSlider(
        value=0.50,
        min=0.00,
        max=1.50,
        step=0.05,
        description="cap_factor",
        continuous_update=False,
    )

    kneed_slider = widgets.FloatSlider(
        value=0.50,
        min=0.00,
        max=1.00,
        step=0.05,
        description="k_need",
        continuous_update=False,
    )

    ui = widgets.VBox([cap_slider, kneed_slider])

    out = widgets.interactive_output(
        plot_scenario,
        {
            "cap_factor": cap_slider,
            "k_need": kneed_slider,
        },
    )

    display(ui, out)
else:
    print("ipywidgets is not installed. The static GitHub figure below will still work.")


## 8. Static GitHub figure

In [ ]:
# Static GitHub-friendly result using the selected scenario from the project
CAP_FACTOR = 0.30
K_NEED = 0.80

P_ev_best, P_grid_best, metrics_best = evaluate_unidirectional(
    cap_factor=CAP_FACTOR,
    k_need=K_NEED
)

time_h = np.arange(len(Pd_mn)) / 60.0

plt.figure(figsize=(12, 5))
plt.plot(time_h, Pd_mn / 1000.0, label="Base demand")
plt.plot(time_h, Ppv / 1000.0, label="PV generation")
plt.plot(time_h, P_ev_best / 1000.0, label="EV charging")
plt.plot(time_h, P_grid_best / 1000.0, label="Grid power")

plt.title("Unidirectional Smart Charging Strategy")
plt.xlabel("Time from start [h]")
plt.ylabel("Power [kW]")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

figure_dir = Path("figures")
figure_dir.mkdir(exist_ok=True)

figure_path = figure_dir / "02_unidirectional_smart_charging.png"
plt.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved figure: {figure_path}")
print("---- Selected scenario metrics ----")
for key, value in metrics_best.items():
    print(f"{key}: {value}")
